# UNITATEA 3 — LABORATOR (Varianta 7)
### „Simularea și validarea unui model probabilist — de la fenomen la decizie”
**Domeniu de aplicare:** Centru de vaccinare  |  **Distribuție vizată:** Poisson

## Pasul 1 — Contextul problemei

Un centru de vaccinare trebuie să-și dimensioneze corect personalul și stocul de doze pentru fiecare oră de program. Dacă fluxul de persoane vaccinate dintr-o oră este supraestimat, resursele (medici, asistenți, doze) rămân neutilizate; dacă este subestimat, se formează cozi, iar eficiența centrului scade.

Conducerea centrului dispune de **numărul de persoane vaccinate în fiecare oră, pe parcursul a 24 de ore de funcționare** și dorește să verifice dacă acest număr poate fi modelat printr-o **distribuție Poisson** (evenimente independente, apărute cu o rată medie constantă). Un astfel de model ar permite calcularea probabilității ca într-o oră dată să fie vaccinate **mai puțin de 5 persoane** (prag considerat de conducere ca indicator de sub-utilizare a centrului).

**Întrebarea:** este distribuția Poisson un model plauzibil pentru numărul de persoane vaccinate pe oră și, dacă da, care este probabilitatea ca într-o oră dată să fie vaccinate mai puțin de 5 persoane, adică $P(X < 5)$?

**Datele necesare:** numărul de persoane vaccinate în fiecare dintre cele 24 de ore de observație (date ilustrative, generate în cod și salvate într-un fișier CSV).

## Pasul 2 — Algoritmul de rezolvare

1. Se generează (sau se importă) datele privind numărul de persoane vaccinate pe oră și se salvează într-un fișier CSV.
2. Se încarcă datele din fișierul CSV.
3. Se calculează media eșantionului observat — pentru distribuția Poisson, aceasta este estimatorul parametrului $\lambda$.
4. Se calculează dispersia eșantionului și se compară cu media (raportul dispersie/medie); pentru o distribuție Poisson autentică, cele două valori ar trebui să fie apropiate.
5. Se simulează un eșantion mare (10 000 de valori) dintr-o distribuție Poisson cu parametrul $\lambda$ estimat la pasul 3.
6. Se reprezintă grafic, prin histograme suprapuse, distribuția simulată și datele observate, pentru o comparație vizuală.
7. Se calculează, pe baza modelului Poisson, probabilitatea ca numărul de persoane vaccinate într-o oră să fie mai mic de 5: $P(X < 5) = P(X \le 4)$.

*Algoritmul este independent de instrument — rămâne valabil în Python, R sau alt limbaj.*

## Pasul 3 — Codul Python

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

**Pasul 2.1** — generarea datelor și salvarea într-un fișier CSV

In [ ]:
date = {
    "ora": list(range(1, 25)),
    "vaccinati": [10, 15, 8, 12, 17, 9, 13, 6, 12, 18, 11, 14,
                  8, 12, 15, 9, 11, 19, 12, 13, 5, 12, 16, 10],
}
df = pd.DataFrame(date)
df.to_csv("vaccinari_orare.csv", index=False)

**Pasul 2.2** — încărcarea datelor din fișierul CSV

In [ ]:
df = pd.read_csv("vaccinari_orare.csv")
x = df["vaccinati"].to_numpy()
print(f"Număr de observații: {len(x)}")

**Pasul 2.3** — estimarea parametrului λ (media eșantionului)

In [ ]:
lambda_estimat = x.mean()
print(f"λ estimat: {lambda_estimat:.2f}")

**Pasul 2.4** — verificarea relației medie ≈ dispersie

In [ ]:
dispersie = x.var(ddof=1)
print(f"Dispersia eșantionului: {dispersie:.2f}")
print(f"Raport dispersie / medie: {dispersie / lambda_estimat:.2f}  (≈ 1 pentru Poisson)")

**Pasul 2.5** — simularea unui eșantion mare din distribuția Poisson

In [ ]:
rng = np.random.default_rng(42)   # seed fix, pentru rezultate reproductibile
simulat = rng.poisson(lam=lambda_estimat, size=10000)

**Pasul 2.6** — comparație grafică (histograme suprapuse)

In [ ]:
# bin-uri centrate pe valorile întregi, comune pentru ambele eșantioane
bins = np.arange(min(x.min(), simulat.min()) - 0.5,
                 max(x.max(), simulat.max()) + 1.5, 1)

plt.figure(figsize=(8, 5))
plt.hist(simulat, bins=bins, density=True, alpha=0.5, label="simulat (Poisson, n=10 000)")
plt.hist(x, bins=bins, density=True, alpha=0.5, label="observat (n=24)")
plt.xlabel("Persoane vaccinate pe oră")
plt.ylabel("Densitate")
plt.title("Centru de vaccinare — observat vs. Poisson simulat")
plt.legend()
plt.savefig("comparatie_poisson.png", dpi=150, bbox_inches="tight")
plt.show()

**Pasul 2.7** — probabilitatea ca într-o oră să fie vaccinate mai puțin de 5 persoane

In [ ]:
prag = 5
prob_sub_prag = stats.poisson.cdf(prag - 1, mu=lambda_estimat)   # P(X <= 4) = P(X < 5)
print(f"P(vaccinați < {prag}) = {prob_sub_prag:.4f}")

# verificare empirică pe eșantionul simulat
print(f"Frecvența empirică în simulare: {(simulat < prag).mean():.4f}")

## Comentariu final

Media eșantionului este $\hat\lambda \approx 11{,}96$ persoane/oră, iar dispersia este $\approx 12{,}82$ — raportul dispersie/medie este de circa 1,07, deci foarte aproape de 1, așa cum se așteaptă pentru o distribuție Poisson autentică. Histogramele observată și simulată se suprapun rezonabil, iar modelul Poisson poate fi considerat plauzibil. Pe baza lui, probabilitatea ca într-o oră să fie vaccinate mai puțin de 5 persoane este $P(X<5)\approx 0{,}0078$ (sub 1%), în concordanță cu datele, unde nicio oră din cele 24 nu a avut sub 5 vaccinări. Limitele: eșantionul este mic (24 de ore), deci histograma observată este neregulată, iar ipoteza de rată constantă poate fi încălcată în practică (ore de vârf vs. ore liniștite); pentru concluzii mai ferme s-ar putea aplica un test de concordanță $\chi^2$ pe un eșantion mai mare.